In [1]:
import os
import logging
import time
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [3]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [4]:
# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("app.log"),  # Archivo de log
        logging.StreamHandler()            # Consola
    ]
)

logger = logging.getLogger(__name__)

In [5]:

def get_products(refined_query):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_embedding`,
          (SELECT @prompt AS content),  -- Aquí usamos el parámetro
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      ML.DISTANCE(
        qe.query_embedding,
        e.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_final_temp` AS d
    INNER JOIN
      `dataton-2024-team-01-cofares.datos_cofares.SalidaEmbeddings_temp` AS e
      ON d.codigo_web = e.title
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """.format(refined_query)
    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", refined_query)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:

        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'


        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')
        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query
        })
    return products

In [6]:
def rerank_products(refined_query, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=refined_query,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [7]:
#FUNCTION CALLING

# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [8]:
PROJECT_ID = "dataton-2024-team-01-cofares"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Importa el modelo de Gemini Flash 1.5
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)


# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)

message_history = []
# Construir el historial de la conversación
historial_conversacion = "\n".join([f"{message['role']}: {message['content']}" for message in message_history])

In [10]:
def refine_query_with_keywords(message_history):

    try:
        # Crear un prompt para el modelo que refine la consulta
        refinement_prompt = """
        Eres un asistente experto en extracción de palabras clave. A continuación, tienes el historial de una conversación:

        Historial de conversación:
        {}

        Extrae las palabras clave más importantes de la consulta del usuario en base al historial y devuélvelas en un formato de texto claro.
        Formato esperado: palabras clave separadas por comas.
        """.format(
            "\n".join([f"{message['role'].capitalize()}: {message['content']}" for message in message_history])
        )

        # Enviar el prompt al modelo para generar el refinamiento
        refinement_response = chat.send_message(refinement_prompt)

        # Obtener la respuesta generada
        refined_query = refinement_response.text.strip()

        logger.info(f"Query refinada generada: {refined_query}")
        return refined_query

    except Exception as e:
        logger.error(f"Error en refine_query_with_keywords: {str(e)}")
        # En caso de error, devolvemos una consulta vacía o un mensaje genérico
        return "consulta vacía"

In [68]:
#INTENTAMOS DEVOLVER LA LISTA DE PRODUCTOS RANKED
def generate_response(prompt):  # Eliminamos el parámetro products
    #chat = multimodal_model.start_chat()

    instruction_prompt = f"""
    # Instrucción
    Eres Cofinder, un asistente farmacéutico experto.\
    Tu tarea consiste en responder eficazmente a las consultas de los profesionales de farmacia.\
    Te proporcionamos una lista de productos procedentes de la base de datos y previamente rankeados por relevancia.\
    Primero debes leer atentamente la entrada del usuario,\
    y luego desarrollar una respuesta basada en los Criterios proporcionados en la sección Producto a continuación.\
    
    # Producto
    ## Definición de la herramienta
    Tienes acceso a una lista de productos de una base de datos de productos de farmacia "{tools}"\
    que han sido reordenados para proporcionar la mejor respuesta posible a la consulta de un profesional de farmacia.\
    Las instrucciones para realizar la tarea de respuesta a una pregunta se proporcionan en la consulta del usuario.\
    
    ## Criterios
    - Si la entrada del profesional de farmacia es un saludo, preséntese cordialmente como Cofinder el asistente de búsqueda.\
        Ejemplos de saludos: «hola», “hola”, “¿Qué tal?”.\
    - Si es necesario, puede pedir detalles aclaratorios para ajustar la búsqueda a resultados eficientes.\
    - Si la entrada solicita búsquedas no relacionadas con productos de farmacia, aclare que ese no es su propósito como asistente de búsqueda de productos de farmacia.\
        Ejemplos de solicitudes no pertinentes: «Quiero la receta de una lasaña», “Quiero pedir una pizza”, “¿Qué tiempo hace hoy?”.\
    - Cuando la entrada sea relevante para activar la búsqueda de productos de farmacia, utiliza "tools" para recibir una lista de productos de farmacia clasificados que ayuden al usuario con su tarea. Acepta la solicitud del usuario y proporciónale la lista de productos sin reescribirla.
    - No sugieras ni añadas productos que no estén en la lista proporcionada por el reranker.

    ### Prompt

        Aquí está la consulta del experto farmacéutico: {prompt}
    """

    try:
        response = chat.send_message(instruction_prompt)
        response.candidates[0].content.parts[0]
        
        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    #refinamos la query
                    refined_query = refine_query_with_keywords(message_history)
                    # Ejecutar búsqueda de productos
                    products = get_products(refined_query)
                    if not products:
                        return "Lo siento, no encontré productos que coincidan con tu búsqueda."
                    
                    ranked_products = rerank_products(refined_query, products)
                    
                    # Devolver directamente la lista de productos
                    return {
                        "type": "product_search",
                        "message": "He encontrado los siguientes productos:",
                        "products": ranked_products["products"]
                    }
                
        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response.text
        }
                    
    except Exception as e:
        return f"Lo siento, ocurrió un error: {str(e)}"

In [11]:
#INTENTAMOS ACTUALIZAR EL HISTORIAL DE MENSAJES
def generate_response(prompt):
    # Agregar el mensaje del usuario al historial
    message_history.append({"role": "user", "content": prompt})

    # Prompt principal
    instruction_prompt = f"""
    # Instrucción
    Eres Cofinder, un asistente farmacéutico experto.\
    Tu tarea consiste en responder eficazmente a las consultas de los profesionales de farmacia.\
    Te proporcionamos una lista de productos procedentes de la base de datos y previamente rankeados por relevancia.\
    Primero debes leer atentamente la entrada del usuario,\
    y luego desarrollar una respuesta basada en los Criterios proporcionados en la sección Producto a continuación.\
    
    # Producto
    ## Definición de la herramienta
    Tienes acceso a una lista de productos de una base de datos de productos de farmacia "{tools}"\
    que han sido reordenados para proporcionar la mejor respuesta posible a la consulta de un profesional de farmacia.\
    Las instrucciones para realizar la tarea de respuesta a una pregunta se proporcionan en la consulta del usuario.\
    
    ## Criterios
    - Si la entrada del profesional de farmacia es un saludo, preséntese cordialmente como Cofinder el asistente de búsqueda.\
        Ejemplos de saludos: «hola», “hola”, “¿Qué tal?».\
    - Si es necesario, puede pedir detalles aclaratorios para ajustar la búsqueda a resultados eficientes.\
    - Si la entrada solicita búsquedas no relacionadas con productos de farmacia, aclare que ese no es su propósito como asistente de búsqueda de productos de farmacia.\
        Ejemplos de solicitudes no pertinentes: «Quiero la receta de una lasaña», “Quiero pedir una pizza”, “¿Qué tiempo hace hoy?».\
    - Cuando la entrada sea relevante para activar la búsqueda de productos de farmacia, utiliza "tools" para recibir una lista de productos de farmacia clasificados que ayuden al usuario con su tarea. Acepta la solicitud del usuario y proporciónale la lista de productos sin reescribirla.
    - No sugieras ni añadas productos que no estén en la lista proporcionada por el reranker.

    ### Prompt

        Aquí está la consulta del experto farmacéutico: {prompt}
    """

    try:
        # Enviar el mensaje al modelo
        response = chat.send_message(instruction_prompt)
        response_text = response.candidates[0].content.parts[0]

        # Agregar la respuesta del modelo al historial
        message_history.append({"role": "assistant", "content": response_text})

        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    # Refinar la query utilizando el historial actualizado
                    refined_query = refine_query_with_keywords(message_history)

                    # Ejecutar búsqueda de productos
                    products = get_products(refined_query)
                    if not products:
                        return {
                            "type": "error",
                            "message": "Lo siento, no encontré productos que coincidan con tu búsqueda."
                        }
                    
                    ranked_products = rerank_products(refined_query, products)
                    
                    # Devolver directamente la lista de productos
                    return {
                        "type": "product_search",
                        "message": "He encontrado los siguientes productos:",
                        "products": ranked_products["products"]
                    }

        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response_text
        }
                    
    except Exception as e:
        logger.error(f"Error en generate_response: {str(e)}")
        return {
            "type": "error",
            "message": f"Lo siento, ocurrió un error: {str(e)}"
        }

In [24]:
# Ejemplo de uso
prompt = "champú, caspa, seca"

In [25]:

products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: APOSAN HOME FLOR PERFUM FLORAL, Descripción: -, Modo de implementación: -, Distancia: 0.8437784911769097
Nombre: APOSAN HOME FLOR PERFUM CITRIC, Descripción: -, Modo de implementación: -, Distancia: 0.8579159375619549
Nombre: APOSAN HOME FLOR PERFUM AMADER, Descripción: -, Modo de implementación: -, Distancia: 0.8699173171169813
Nombre: INFLA HEAL 90CAP, Descripción: -, Modo de implementación: -, Distancia: 0.8722335689754138
Nombre: APOSAN ESENCIA HOJAS DE HIGUERA 10 MILILITROS, Descripción: -, Modo de implementación: -, Distancia: 0.8722952625348821
Nombre: HISOPO REAL AC ESEN 5ML, Descripción: 5 ml, Modo de implementación: -, Distancia: 0.875396570909917
Nombre: HINOJO BIO AC ESENC 5ML, Descripción: Originario de la cuenca mediterránea, el hinojo dulce es una apiácea (umbelíferas) que puede llegar a alcanzar hasta 2 metros de altura. Se caracteriza por umbelas de flores amarillas y hojas pinadas y plumosas, así como de semillas oblongas características d

In [26]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: HINOJO BIO AC ESENC 5ML, Descripción: Originario de la cuenca mediterránea, el hinojo dulce es una apiácea (umbelíferas) que puede llegar a alcanzar hasta 2 metros de altura. Se caracteriza por umbelas de flores amarillas y hojas pinadas y plumosas, así como de semillas oblongas características de la familia botánica. Consejos & trucos: 2 gotas AEQT Hinojo dulce + 2 gotas AEQT Estragón + 2 gotas AEQT Salvia officinalis + 2 gotas AEQT Manzanilla en 5 gotas AV Nuez de albaricoque para masajear la parte inferior del vientre y mejorar las reglas dolorosas y los problemas de la menopausia. De esta misma mezcla se pueden tomar 4 gotas en un poco de miel, vía oral, 2 veces al día, para las mismas indicaciones durante el tiempo necesario hasta que se aprecie una mejoría., Modo de implementación: 2 gotas 3 veces al día sobre un soporte neutro ( Miel, azúcar de caña o aceite vegetal). Apto para uso cutáneo., Distancia: 0.8790141021599998
Nombre: ANPAHI FORTE 20 AMPOL

In [23]:
response_text = generate_response(prompt)  # Generar la respuesta
print(response_text)

2024-11-19 23:03:22,480 - INFO - Query refinada generada: crema hidratante, manos, resecas


{'type': 'product_search', 'message': 'He encontrado los siguientes productos:', 'products': [{'codigo_web': '100942', 'nombre': 'CONTORNO OJOS PRO-COLLAG 15ML', 'codigo_nacional': '1009425', 'descripcion': 'Potencia la producción de colágeno en piel. Hidrata, nutre y regenera. Proporciona antioxidantes naturales para combatir el paso del tiempo. Acción antimanchas y protección uva-uvb natural, que actúa en el interior de la piel gracias al ácido hialurónico natural. 15 ml', 'modo_implementacion': 'Aplicar sobre la piel limpia del contorno de los ojos y párpados realizando un suave masaje desde el centro de los ojos hacia las sienes. CONSERVACIÓN: conservar al abrigo de la luz y del calor. USO EXTERNO. TESTADO BAJO CONTROL DERMATOLÓGICO. Mantener fuera del alcance de los niños. Evitar el contacto con los ojos y la boca. No ingerir.', 'imagen_url': 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/reto_cofares/100942.jpg', 'distance_to_query': 